# 01 – Cleaning HM Land Registry Price Paid Data (2021)

**Goal:** turn the raw Price Paid file into a clean table of normal residential sales, ready for PostgreSQL, analysis and modelling.

Run the cells from top to bottom. Every cleaning decision is written down with the reason, so you can explain it in an interview.

**Before you start:** put `pp-2021.csv` in `data/raw/`. Do **not** open and save it in Excel.

## Step 1 – Load the file and add the headers

The raw file has **no header row**, so the first sale would be read as column names. We supply the 16 names ourselves (from the gov.uk "Explanations of column headers" page).

We read every column as text (`dtype=str`) first. This stops pandas guessing types, e.g. dropping leading zeros or turning blanks into `NaN` in different ways per column. We convert types deliberately in Step 3.

In [4]:
import pandas as pd
RAW_FILE = "/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Raw/pp-2021.csv"
CLEAN_FILE = "/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2021-clean.csv"

COLUMNS = [
    "transaction_id",   # unique ID for each sale
    "price",            # sale price in £
    "date_of_transfer", # date the sale completed
    "postcode",
    "property_type",    # D=Detached, S=Semi-detached, T=Terraced, F=Flat/maisonette, O=Other
    "new_build",        # Y=new build, N=existing property
    "tenure",           # F=Freehold, L=Leasehold
    "paon",             # Primary Addressable Object Name (house number or name)
    "saon",             # Secondary Addressable Object Name (flat number etc.)
    "street",
    "locality",
    "town",
    "district",
    "county",
    "ppd_category",     # A=standard market sale, B=additional (repossessions, buy-to-let, company sales...)
    "record_status",    # A=addition, C=change, D=delete (only matters for monthly update files)
]

df = pd.read_csv(RAW_FILE, header=None, names=COLUMNS, dtype=str, keep_default_na=False)
print(f"Rows loaded: {len(df):,}")
df.head()

Rows loaded: 1,281,821


,transaction_id,price,date_of_transfer,postcode,property_type,new_build,tenure,paon,saon,street,locality,town,district,county,ppd_category,record_status
0,{E53EDD2E-C6F0-83EC-E053-6B04A8C03A59},70000,2021-06-11 00:00,HX2 8SN,T,N,F,16,,CLOUGH LANE,MIXENDEN,HALIFAX,CALDERDALE,WEST YORKSHIRE,B,A
1,{E53EDD2E-C6F1-83EC-E053-6B04A8C03A59},150000,2021-11-08 00:00,LS27 9AL,O,N,F,RUSSELL HOUSE,3,HIGH STREET,MORLEY,LEEDS,LEEDS,WEST YORKSHIRE,B,A
2,{E53EDD2E-C6F3-83EC-E053-6B04A8C03A59},430000,2021-10-29 00:00,LS6 1BU,D,N,F,10,,MANOR TERRACE,HEADINGLEY,LEEDS,LEEDS,WEST YORKSHIRE,B,A
3,{E53EDD2E-C6F4-83EC-E053-6B04A8C03A59},135000,2021-10-26 00:00,LS10 1LP,F,N,L,MCCLINTOCK HOUSE,APARTMENT 32,THE BOULEVARD,,LEEDS,LEEDS,WEST YORKSHIRE,B,A
4,{E53EDD2E-C704-83EC-E053-6B04A8C03A59},131000,2021-12-21 00:00,HD2 2SN,O,N,F,244,,HALIFAX OLD ROAD,BIRKBY,HUDDERSFIELD,KIRKLEES,WEST YORKSHIRE,B,A


## Step 2 – Explore data quality before changing anything

Always look first, then decide. We check blanks, the category counts and the price range.

In [5]:
# Blank values per column
blanks = (df == "").sum().to_frame("blank_count")
blanks["blank_pct"] = (blanks["blank_count"] / len(df) * 100).round(1)
blanks

,blank_count,blank_pct
transaction_id,0,0.0
price,0,0.0
date_of_transfer,0,0.0
postcode,3889,0.3
property_type,0,0.0
new_build,0,0.0
tenure,0,0.0
paon,0,0.0
saon,1118877,87.3
street,25678,2.0


**What this shows:** `saon` and `locality` are mostly blank. That's normal (most homes aren't flats, and many addresses have no locality), so we keep them and don't treat them as errors. The column that matters is `postcode`: a sale without one can't be matched to a region or location later.

In [6]:
for col in ["property_type", "new_build", "tenure", "ppd_category", "record_status"]:
    print(col, df[col].value_counts().to_dict())

property_type {'T': 344848, 'S': 335581, 'D': 314300, 'F': 216703, 'O': 70389}
new_build {'N': 1146056, 'Y': 135765}
tenure {'F': 996942, 'L': 284879}
ppd_category {'A': 1086854, 'B': 194967}
record_status {'A': 1281821}


In [7]:
# Price summary (converted to numbers just for this check)
pd.to_numeric(df["price"]).describe().apply(lambda x: f"{x:,.0f}")

count      1,281,821
mean         389,557
std        1,764,145
min                1
25%          172,000
50%          269,995
75%          412,525
max      900,000,000
Name: price, dtype: str

**What this shows:** the minimum price is £1 and the maximum is £900 million. Neither is a normal house sale, so we need to remove extreme values. Also, every row has `record_status = A`, so there are no changes or deletions to apply in a yearly file.

## Step 3 – Fix data types

- `price` → whole number
- `date_of_transfer` → a real date (the time part is always `00:00`, so we drop it)
- blank text → proper missing values (`NA`) so SQL gets `NULL` later

In [8]:
df["price"] = df["price"].astype("int64")
df["date_of_transfer"] = pd.to_datetime(df["date_of_transfer"], format="%Y-%m-%d %H:%M").dt.date
df = df.rename(columns={"date_of_transfer": "sale_date"})

# Turn empty strings into missing values
df = df.replace("", pd.NA)

# Sanity check: every date should be in 2021
print(df["sale_date"].min(), "to", df["sale_date"].max())
df.dtypes

2021-01-01 to 2021-12-31


transaction_id       str
price              int64
sale_date         object
postcode             str
property_type        str
new_build            str
tenure               str
paon                 str
saon                 str
street               str
locality             str
town                 str
district             str
county               str
ppd_category         str
record_status        str
dtype: object

## Step 4 – Apply the cleaning rules

We keep a **cleaning log** so we can report exactly how many rows each rule removed. Put this table in your README.

In [9]:
cleaning_log = [("Raw file", len(df), 0)]

def apply_rule(data, keep_mask, description):
    before = len(data)
    data = data[keep_mask]
    cleaning_log.append((description, len(data), before - len(data)))
    print(f"{description:<55} removed {before - len(data):>8,}   remaining {len(data):>10,}")
    return data

### Rule 1 – Keep only standard market sales (`ppd_category = 'A'`)

Category B includes repossessions, buy-to-let purchases with a mortgage, transfers to companies and other non-standard deals. Their prices don't reflect a normal buyer paying market value, which would confuse the model.

In [10]:
df = apply_rule(df, df["ppd_category"] == "A", "Keep standard market sales (category A)")

Keep standard market sales (category A)                 removed  194,967   remaining  1,086,854


### Rule 2 – Remove property type 'O' (Other)

"Other" is a mix of things that aren't a normal home (e.g. commercial or mixed-use).

*Interesting finding:* in 2021 **every** 'O' sale is already category B, so Rule 1 removed them all and this rule removes 0 rows. We keep the rule anyway so the code still works on other years. This is a good detail to mention in an interview.

In [11]:
df = apply_rule(df, df["property_type"] != "O", "Remove property type O (Other)")

Remove property type O (Other)                          removed        0   remaining  1,086,854


### Rule 3 – Remove sales with no postcode

Without a postcode we can't get the region, local authority, coordinates or deprivation score.

In [12]:
df = apply_rule(df, df["postcode"].notna(), "Remove sales with no postcode")

Remove sales with no postcode                           removed      201   remaining  1,086,653


### Rule 4 – Remove extreme prices (below £10,000 or above £5,000,000)

Prices under £10k are almost never a real open-market home sale (often a share of a property, or a data error). Above £5m is a tiny number of ultra-prime homes that would dominate errors. These cut-offs are a judgement call: say so, and explain why you chose them.

In [13]:
df = apply_rule(df, df["price"].between(10_000, 5_000_000), "Remove prices outside £10k–£5m")

Remove prices outside £10k–£5m                          removed      582   remaining  1,086,071


### Rule 5 – Remove duplicate sales

`transaction_id` is unique, so there are no exact duplicates. But some sales appear **twice with different IDs**: same address, same date, same price. These are the same sale recorded more than once, so we keep only the first.

In [14]:
print("Duplicate transaction IDs:", df["transaction_id"].duplicated().sum())

dup_key = ["postcode", "paon", "saon", "price", "sale_date"]
print("Same address + date + price:", df.duplicated(subset=dup_key).sum())

# Look at a few examples before deleting
df[df.duplicated(subset=dup_key, keep=False)].sort_values(dup_key).head(6)[
    ["postcode", "paon", "saon", "street", "price", "sale_date", "property_type"]
]

Duplicate transaction IDs: 0
Same address + date + price: 1777


,postcode,paon,saon,street,price,sale_date,property_type
13720,AL7 2AN,118,NaN,BROOKSFIELD,350000,2021-06-30,S
41485,AL7 2AN,118,NaN,BROOKSFIELD,350000,2021-06-30,S
895581,AL7 2AN,118,NaN,BROOKSFIELD,350000,2021-06-30,T
103436,B11 3LQ,271,NaN,PERCY ROAD,157000,2021-09-17,T
109029,B11 3LQ,271,NaN,PERCY ROAD,157000,2021-09-17,T
685475,B13 0QB,60,NaN,CHAMBERLAIN ROAD,346500,2021-12-16,S


In [15]:
df = apply_rule(df, ~df.duplicated(subset=dup_key, keep="first"), "Remove duplicate sales (same address, date, price)")

Remove duplicate sales (same address, date, price)      removed    1,777   remaining  1,084,294


## Step 5 – Tidy text and add useful columns

- **postcode_district**: the first half of the postcode (e.g. `E17`). This is what users will type into the app.
- **postcode_area**: the letters only (e.g. `E`, `SW`, `LS`). Useful for bigger groupings.
- **sale_year / sale_month**: for trends and seasonality.
- Readable labels for the one-letter codes, which make charts easier to read.
- Drop `ppd_category` and `record_status`: after cleaning they have only one value, so they carry no information.

In [16]:
df["postcode"] = df["postcode"].str.strip().str.upper()
df["postcode_district"] = df["postcode"].str.split(" ").str[0]
df["postcode_area"] = df["postcode_district"].str.extract(r"^([A-Z]+)", expand=False)

sale_dt = pd.to_datetime(df["sale_date"])
df["sale_year"] = sale_dt.dt.year
df["sale_month"] = sale_dt.dt.month

df["property_type_label"] = df["property_type"].map(
    {"D": "Detached", "S": "Semi-detached", "T": "Terraced", "F": "Flat"})
df["tenure_label"] = df["tenure"].map({"F": "Freehold", "L": "Leasehold"})

df = df.drop(columns=["ppd_category", "record_status"])
df.head()

,transaction_id,price,sale_date,postcode,property_type,new_build,tenure,paon,saon,street,locality,town,district,county,postcode_district,postcode_area,sale_year,sale_month,property_type_label,tenure_label
28,{E7B085FC-8E46-7E31-E053-6C04A8C0E67F},195000,2021-08-27,PE38 9GH,T,N,F,82,NaN,BENNETT STREET,NaN,DOWNHAM MARKET,KING'S LYNN AND WEST NORFOLK,NORFOLK,PE38,PE,2021,8,Terraced,Freehold
29,{E7B085FC-8E4F-7E31-E053-6C04A8C0E67F},200000,2021-08-27,NR7 0PB,S,N,F,3,NaN,THORPE CLOSE,NaN,NORWICH,BROADLAND,NORFOLK,NR7,NR,2021,8,Semi-detached,Freehold
30,{E7B085FC-8E5F-7E31-E053-6C04A8C0E67F},190000,2021-12-17,PE30 2LF,T,N,F,89,NaN,COLUMBIA WAY,NaN,KING'S LYNN,KING'S LYNN AND WEST NORFOLK,NORFOLK,PE30,PE,2021,12,Terraced,Freehold
31,{E7B085FC-8E7F-7E31-E053-6C04A8C0E67F},225000,2021-08-20,NR17 1XX,S,N,F,1A,NaN,TEASEL ROAD,NaN,ATTLEBOROUGH,BRECKLAND,NORFOLK,NR17,NR,2021,8,Semi-detached,Freehold
32,{E7B085FC-8E86-7E31-E053-6C04A8C0E67F},305000,2021-09-28,NR4 7JY,T,N,F,14,NaN,TULIP GARDENS,CRINGLEFORD,NORWICH,SOUTH NORFOLK,NORFOLK,NR4,NR,2021,9,Terraced,Freehold


## Step 6 – Final checks

In [17]:
assert df["transaction_id"].is_unique
assert df["postcode"].notna().all()
assert df["price"].between(10_000, 5_000_000).all()
assert df["property_type_label"].notna().all()
print("All checks passed")

log = pd.DataFrame(cleaning_log, columns=["step", "rows_remaining", "rows_removed"])
log["pct_of_raw_kept"] = (log["rows_remaining"] / log.loc[0, "rows_remaining"] * 100).round(1)
log

All checks passed


,step,rows_remaining,rows_removed,pct_of_raw_kept
0,Raw file,1281821,0,100.0
1,Keep standard market sales (category A),1086854,194967,84.8
2,Remove property type O (Other),1086854,0,84.8
3,Remove sales with no postcode,1086653,201,84.8
4,Remove prices outside £10k–£5m,1086071,582,84.7
5,"Remove duplicate sales (same address, date, pr...",1084294,1777,84.6


In [18]:
# Median price by property type – a quick check that the numbers look sensible
df.groupby("property_type_label")["price"].agg(["count", "median"]).sort_values("median")

,count,median
property_type_label,,
Terraced,291422,220000.0
Flat,180432,237500.0
Semi-detached,310304,248000.0
Detached,302136,389950.0


## Step 7 – Save the clean file

This file (with a header row) is what you load into PostgreSQL. It lives in `data/clean/`, which `.gitignore` keeps off GitHub.

In [19]:
import os
os.makedirs(os.path.dirname(CLEAN_FILE), exist_ok=True)
df.to_csv(CLEAN_FILE, index=False)
print(f"Saved {len(df):,} rows to {CLEAN_FILE}")

Saved 1,084,294 rows to /Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2021-clean.csv


In [20]:
   import os
   print(os.path.abspath(CLEAN_FILE))

/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2021-clean.csv


In [21]:
import os
print(os.path.abspath("data/clean/pp-2021-clean.csv"))
print("File exists:", os.path.exists("data/clean/pp-2021-clean.csv"))

/Users/yuvarajkotte/PycharmProjects/JupyterProject1/uk_house_prices/data/clean/pp-2021-clean.csv
File exists: False


In [22]:
import glob
project = "/Users/yuvarajkotte/PycharmProjects/JupyterProject1"

print("Clean file:", glob.glob(project + "/**/pp-2021-clean.csv", recursive=True))
print("Raw file:  ", glob.glob(project + "/**/pp-2021.csv", recursive=True))

Clean file: ['/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2021-clean.csv']
Raw file:   ['/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Raw/pp-2021.csv']
